# 2026 FIELD CAMP 주제 1 — Phase 2 관중 입장 수익 최대화 스케줄링
## MILP 코드 제출 양식 (.ipynb) — **재정식화판 (v2)**

**팀명:** 비원

---
### v1 대비 변경점

| 항목 | v1 | v2 |
|:--|:--|:--|
| 조건 ⑥ 선형화 | McCormick $y_{tvws}$ (연속 35,640개) | 조건부 등식 big-M $g_{ts}$ (연속 440개) |
| 모델 크기 | 115,995행 × 44,200열 | **16,555행 × 9,000열** (행 7.0배↓) |
| LP 이완값 | 81,000.0617 | 81,000.0617 (**완전히 동일**) |
| 제약식 번호 | (1)~(28) | (1)~(26) |

**재정식화 근거 (실측).**

1. 조건 ⑥을 통째로 제거한 ①~⑤ 모델의 순수 LP값 = **81,000.061725**.
2. v2 모델(①~⑥, big-M)의 순수 LP값 = **81,000.061725** — 소수점 6자리까지 동일.
3. v1의 root bound(McCormick + RLT 1,682개 등 모든 cut 적용 후) = 80,992.9366.

즉 $y$ 블록 약 107,000행이 만들어낸 bound 개선은 **총 7.12 (0.0088%)** 에 불과한 반면,
노드당 LP는 감당 못 할 만큼 커졌다(노드당 simplex 2,095회, 8시간에 42,930노드 = 1.5노드/초).
아래 (16)(17)은 $y$ 없이 **동일한 이동거리를 정확히** 표현한다
(v1 최선해 78,586을 v2에 대입해 $L_t$ 10개가 직접 계산값과 전부 일치함을 확인).

- 각 제약식 = 마크다운(번호 + LaTeX + 범위 + 설명) + 코드 한 쌍. 번호는 **docx와 1:1 일치**.
- **출력물:** 스케줄 `비원_Phase2_schedule.csv`, 실행 로그 `비원_phase2_log.txt` (정오표 1).
- 제출 전 **Restart & Run All**.

## 0. 환경 · 데이터

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from pathlib import Path
from collections import Counter

DATA_DIR = Path(".")
if not (DATA_DIR / "p_phase2_series.csv").exists():
    for _d in (Path(".."), Path("../..")):
        if (_d / "p_phase2_series.csv").exists():
            DATA_DIR = _d; break
print("DATA_DIR =", DATA_DIR.resolve())

p_series   = pd.read_csv(DATA_DIR / "p_phase2_series.csv",   encoding="utf-8-sig")
team_venue = pd.read_csv(DATA_DIR / "team_venue_phase2.csv", encoding="utf-8-sig")
D_df       = pd.read_csv(DATA_DIR / "D_phase2.csv",          encoding="utf-8-sig", index_col=0)
print("로드 완료 |", p_series.shape, team_venue.shape, D_df.shape)
WLS_ENV = None

DATA_DIR = C:\Users\3124s\Desktop\필드_2026
로드 완료 | (4050, 4) (10, 4) (9, 9)


## 1. Sets & Indices *(docx Table 1과 1:1 일치)*

| 기호 | 정의 | 크기 |
|:--|:--|:--|
| $T$ | 구단 집합 | 10 |
| $S$ | 시리즈 집합 | 45 |
| $T^{order}$ | 순서쌍 $(i,j),\ i\neq j$ (i=홈) | 90 |
| $T^{unorder}$ | 비순서쌍 $\{i,j\}$ | 45 |
| $V$ | 구장 집합 (Metro_Field 공유로 9개) | 9 |
| $B_1,B_2,B_3$ | 시리즈 구간 1–18 / 19–36 / 37–45 | 18/18/9 |
| $S^{W}$ | 주말(홀수 번째) 시리즈 | 23 |
| $S^{cons}$ | 연속 4시리즈 창의 시작점 $\{1,\dots,42\}$ | 42 |
| $S^{move}$ | 이동 구간 시작점 $\{1,\dots,44\}$ | 44 |

In [2]:
T = list(range(1, 11)); S = list(range(1, 46))
Torder   = [(i, j) for i in T for j in T if i != j]    # 90
Tunorder = [(i, j) for i in T for j in T if i < j]     # 45
V = list(D_df.index)                                   # 9 (Metro_Field 공유)
B1 = list(range(1, 19)); B2 = list(range(19, 37)); B3 = list(range(37, 46))
SW = [s for s in S if s % 2 == 1]                      # 주말 = 홀수 번째, 23개
Scons = list(range(1, 43))                             # 연속 4시리즈 창
Smove = list(range(1, 45))                             # s -> s+1 이동
print("집합 OK | Torder", len(Torder), "| Tunorder", len(Tunorder), "| V", len(V), "| SW", len(SW))

집합 OK | Torder 90 | Tunorder 45 | V 9 | SW 23


## 2. Parameters

| 기호 | 의미 | 값/출처 |
|:--|:--|:--|
| $p_{ijs}$ | $i$홈 vs $j$원정 시리즈를 $s$에 편성할 때 수익 | `p_phase2_series.csv` (4,050개) |
| $d_{vw}$ | 구장 $v\to w$ 이동거리 | `D_phase2.csv` |
| $e_{tv}$ | 구단 $t$의 홈구장이 $v$이면 1 | `team_venue_phase2.csv` |
| $n^{meet}$ | 두 팀 간 시즌 대결 시리즈 수 | 5 (규칙 ③) |
| $c^{max}$ | 연속 4시리즈 내 최대 동일 홈/원정 수 | 3 (규칙 ⑤) |
| $H^{low},H^{high}$ | 시즌 홈 시리즈 하/상한 | 22 / 23 (규칙 ④) |
| $W^{low},W^{high}$ | 주말 홈 시리즈 하/상한 | 11 / 12 (규칙 ④) |
| $D^{max}$ | 이동거리 편차 한도 | **4500** (FAQ Q3) |
| $M$ | big-M | $\max_{v,w} d_{vw}=445$ |

**개막 호스트 집합 $R$ 유도식 (21)** — 직전 시즌 상위 5팀 $T^{5}$ (FAQ Q2: `team_id` = 전년도 순위).
Metro_Field 공유 구단 $C=\{1,2\}$가 모두 상위 5에 들면 순위가 낮은 쪽(Rams, id 2)을 제외하고
6위(Chiefs, id 6)가 대체한다.

$$R = \begin{cases}
(T^{5}\setminus\{\arg\max_{t\in T^{5}\cap C}\text{rank}(t)\})\cup\{t^{(6)}\}, & |T^{5}\cap C|\ge 2\\[2pt]
T^{5}, & \text{otherwise}
\end{cases}\tag{21}$$

$\Rightarrow R=\{1,3,4,5,6\}$. 최적화 제약이 아니라 **입력 단계에서 사전 계산되는 유도식**이다.

In [3]:
p    = {(int(r.home_team_id), int(r.away_team_id), int(r.series)): float(r.p) for r in p_series.itertuples()}
ven  = dict(zip(team_venue.team_id, team_venue.venue_name))
e    = {(t, v): (1 if ven[t] == v else 0) for t in T for v in V}
dist = {(a, b): float(D_df.loc[a, b]) for a in D_df.index for b in D_df.columns}

n_meet, c_max = 5, 3
H_lo, H_hi, w_lo, w_hi, D_max = 22, 23, 11, 12, 4500
BIGM = max(dist.values())                       # = 445, 거리행렬 최댓값

# (21) 개막 호스트 R 유도 — 하드코딩이 아니라 데이터에서 계산
_share = Counter(ven[t] for t in T)
shared = [v for v, c in _share.items() if c > 1]         # 공유 구장 = Metro_Field
C      = sorted([t for t in T if ven[t] in shared])       # C = {1, 2}
rank   = {t: t for t in T}                                # FAQ Q2: team_id = 전년도 순위
order  = sorted(T, key=lambda t: rank[t]); top5 = order[:5]
mit    = [t for t in top5 if t in C]
R = sorted([t for t in top5 if t != max(mit, key=lambda t: rank[t])] + [order[5]]) if len(mit) >= 2 else sorted(top5)

print("파라미터 OK | p", len(p), "| M", BIGM, "| C", C, "| R", R)

파라미터 OK | p 4050 | M 445.0 | C [1, 2] | R [1, 3, 4, 5, 6]


## 3. Decision Variables

| 기호 | 종류 | 개수 | 정의 |
|:--|:--|:--|:--|
| $x_{ijs}$ | binary | 4,050 | 시리즈 $s$에 $i$홈 vs $j$원정 편성이면 1 |
| $h_{ts}$ | binary | 450 | 구단 $t$가 시리즈 $s$에 홈이면 1 — 식 (2) |
| $z_{tvs}$ | binary | 4,050 | 구단 $t$가 시리즈 $s$에 구장 $v$에 있으면 1 — 식 (14) |
| $g_{ts}$ | continuous $[0,M]$ | 440 | 시리즈 $s\to s+1$ 이동거리 — 식 (16)(17) |
| $L_{t}$ | continuous $\ge 0$ | 10 | 구단 $t$의 시즌 총 이동거리 — 식 (18) |

> v1의 $y_{tvws}$ (연속 35,640개)는 $g_{ts}$ 440개로 대체되었다.

In [4]:
m = gp.Model("phase2_v2", env=WLS_ENV) if WLS_ENV else gp.Model("phase2_v2")
x = m.addVars(Torder, S,     vtype=GRB.BINARY,     name="x")
h = m.addVars(T, S,          vtype=GRB.BINARY,     name="h")
z = m.addVars(T, V, S,       vtype=GRB.BINARY,     name="z")
g = m.addVars(T, Smove, lb=0.0, ub=BIGM, vtype=GRB.CONTINUOUS, name="g")
L = m.addVars(T,        lb=0.0,          vtype=GRB.CONTINUOUS, name="L")
m.update(); print("변수:", m.NumVars, "(v1: 44,200)")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2847604
Academic license 2847604 - for non-commercial use only - registered to bi___@pusan.ac.kr
변수: 9000 (v1: 44,200)


## 4. Objective

시즌 전체 관중 입장 수익의 합을 최대화한다.

$$\text{Maximize } Z=\sum_{(i,j)\in T^{order}}\sum_{s\in S} p_{ijs}\,x_{ijs}$$

In [5]:
m.setObjective(gp.quicksum(p[i, j, s] * x[i, j, s] for (i, j) in Torder for s in S), GRB.MAXIMIZE)
m.update(); print("목적함수 OK")

목적함수 OK


## 5. Constraints (docx 번호 (1)~(26)과 일치)

### 1) 시리즈 편성 기본 구조 — 조건 ①
10개 구단이 매 시리즈 5개 매치업으로 빠짐없이 편성되고, 홈 여부 지시변수 $h$를 $x$에 연결한다.

$$\sum_{j\neq t}\left(x_{tjs}+x_{jts}\right)=1\qquad \forall t\in T,\ s\in S \tag{1}$$
$$h_{ts}=\sum_{j\neq t}x_{tjs}\qquad \forall t\in T,\ s\in S \tag{2}$$

In [6]:
m.addConstrs((gp.quicksum(x[t, j, s] for j in T if j != t) + gp.quicksum(x[j, t, s] for j in T if j != t) == 1
              for t in T for s in S), name="c1")
m.addConstrs((h[t, s] == gp.quicksum(x[t, j, s] for j in T if j != t)
              for t in T for s in S), name="c2")
print("① 추가")

① 추가


### 2) 개막 · 올스타 직후 4연전 — 조건 ②
개막 시리즈(1번)의 홈 개최권은 $R$ (식 21)이 갖고, 31번 시리즈는 개막 시리즈의
매치업과 홈·원정 배치를 **그대로** 반복한다.

$$h_{t,1}=1\qquad \forall t\in R \tag{3}$$
$$x_{ij,31}=x_{ij,1}\qquad \forall (i,j)\in T^{order} \tag{4}$$

In [7]:
m.addConstrs((h[t, 1] == 1 for t in R), name="c3")
m.addConstrs((x[i, j, 31] == x[i, j, 1] for (i, j) in Torder), name="c4")
print("② 추가")

② 추가


### 3) 매치업 횟수 · 간격 — 조건 ③
같은 두 팀은 시즌 동안 정확히 5개 시리즈에서 만나고, 어느 연속 4시리즈 구간을 잘라 보아도
같은 매치업은 최대 1개다(재대결까지 최소 3시리즈 간격).

$$\sum_{s\in S}\left(x_{ijs}+x_{jis}\right)=n^{meet}\qquad \forall (i,j)\in T^{unorder} \tag{5}$$
$$\sum_{s'=s}^{s+3}\left(x_{ijs'}+x_{jis'}\right)\le 1\qquad \forall (i,j)\in T^{unorder},\ s\in S^{cons} \tag{6}$$

In [8]:
m.addConstrs((gp.quicksum(x[i, j, s] + x[j, i, s] for s in S) == n_meet
              for (i, j) in Tunorder), name="c5")
m.addConstrs((gp.quicksum(x[i, j, sp] + x[j, i, sp] for sp in range(s, s + 4)) <= 1
              for (i, j) in Tunorder for s in Scons), name="c6")
print("③ 추가")

③ 추가


### 4) 홈 · 원정 배분과 총량 — 조건 ④
1–18 구간과 19–36 구간에서 각 팀은 9개 상대 각각과 홈 1회 · 원정 1회를 치른다
(→ 상대별 다섯 번째 시리즈는 자동으로 $B_3$에 놓인다). 시즌 전체 홈은 22~23개,
그중 주말 홈은 11~12개다.

$$\sum_{s\in B_1}x_{ijs}=1 \qquad \forall (i,j)\in T^{order} \tag{7}$$
$$\sum_{s\in B_2}x_{ijs}=1 \qquad \forall (i,j)\in T^{order} \tag{8}$$
$$H^{low}\le\sum_{s\in S}h_{ts}\le H^{high} \qquad \forall t\in T \tag{9}$$
$$W^{low}\le\sum_{s\in S^{W}}h_{ts}\le W^{high} \qquad \forall t\in T \tag{10}$$

In [9]:
m.addConstrs((gp.quicksum(x[i, j, s] for s in B1) == 1 for (i, j) in Torder), name="c7")
m.addConstrs((gp.quicksum(x[i, j, s] for s in B2) == 1 for (i, j) in Torder), name="c8")
m.addConstrs((gp.quicksum(h[t, s] for s in S)  >= H_lo for t in T), name="c9lo")
m.addConstrs((gp.quicksum(h[t, s] for s in S)  <= H_hi for t in T), name="c9hi")
m.addConstrs((gp.quicksum(h[t, s] for s in SW) >= w_lo for t in T), name="c10lo")
m.addConstrs((gp.quicksum(h[t, s] for s in SW) <= w_hi for t in T), name="c10hi")
print("④ 추가")

④ 추가


### 5) 연속 제한과 구장 사용 — 조건 ⑤
홈도 원정도 4연속은 불가(=연속 4시리즈 창에서 홈이 최대 3개, 최소 1개).
같은 구장에서 같은 시리즈에 두 경기가 열릴 수 없으므로 Metro_Field 공유 구단 $C$는
동시에 홈이 될 수 없다.

$$\sum_{s'=s}^{s+3}h_{ts'}\le c^{max}\qquad \forall t\in T,\ s\in S^{cons} \tag{11}$$
$$\sum_{s'=s}^{s+3}\left(1-h_{ts'}\right)\le c^{max}\qquad \forall t\in T,\ s\in S^{cons} \tag{12}$$
$$\sum_{t\in C}h_{ts}\le 1\qquad \forall s\in S \tag{13}$$

In [10]:
m.addConstrs((gp.quicksum(h[t, sp] for sp in range(s, s + 4)) <= c_max
              for t in T for s in Scons), name="c11")
m.addConstrs((gp.quicksum(1 - h[t, sp] for sp in range(s, s + 4)) <= c_max
              for t in T for s in Scons), name="c12")
m.addConstrs((gp.quicksum(h[t, s] for t in C) <= 1 for s in S), name="c13")
print("⑤ 추가")

⑤ 추가


### 6) 이동거리 편차 — 조건 ⑥ *(핵심 재정식화)*

**(14)(15) 위치 변수.** 구단 $t$는 홈이면 자기 구장, 원정이면 상대 홈구장에 있다.
$z$는 $x,h$에 의해 완전히 결정되며 (15)에 의해 one-hot이다.

$$z_{tvs}=e_{tv}\,h_{ts}+\!\!\sum_{i\neq t:\,e_{iv}=1}\!\! x_{its}\qquad \forall t\in T,\ v\in V,\ s\in S \tag{14}$$
$$\sum_{v\in V}z_{tvs}=1\qquad \forall t\in T,\ s\in S \tag{15}$$

**(16)(17) 구간 이동거리 — 조건부 등식의 big-M 선형화.**
(15)에 의해 $z_{t\cdot s}$는 one-hot이므로, "현재 구장이 $v$일 때
$g_{ts}=\sum_w d_{vw}z_{t,w,s+1}$" 이라는 조건부 등식 하나로 이동거리가 **정확히** 결정된다.
이를 양방향 부등식으로 쓴 것이 (16)(17)이다.

$$g_{ts}-\sum_{w\in V}d_{vw}\,z_{t,w,s+1}\ \le\ M\left(1-z_{tvs}\right)\qquad \forall t\in T,\ v\in V,\ s\in S^{move} \tag{16}$$
$$\sum_{w\in V}d_{vw}\,z_{t,w,s+1}-g_{ts}\ \le\ M\left(1-z_{tvs}\right)\qquad \forall t\in T,\ v\in V,\ s\in S^{move} \tag{17}$$

$z_{tvs}=1$인 유일한 $v$에서만 등식이 강제되고, 나머지 $v$에서는
$0\le g_{ts}\le M$, $0\le\sum_w d_{vw}z\le M$ 이므로 항상 이완된다.
$M=\max_{v,w}d_{vw}=445$로 매우 작아 LP 이완이 느슨해지지 않는다.

**(18)~(20) 총 이동거리와 편차.** 이동은 시리즈 사이에서만 발생하고(집결 이동 제외),
규제 대상은 총합이 아니라 **구단 간 차이**다.

$$L_t=\sum_{s\in S^{move}}g_{ts}\qquad \forall t\in T \tag{18}$$
$$L_t-L_u\le D^{max}\qquad \forall (t,u)\in T^{unorder} \tag{19}$$
$$L_u-L_t\le D^{max}\qquad \forall (t,u)\in T^{unorder} \tag{20}$$

In [11]:
# (14)(15) 위치 변수 정의
m.addConstrs((z[t, v, s] == e[t, v] * h[t, s] + gp.quicksum(x[i, t, s] for i in T if i != t and e[i, v] == 1)
              for t in T for v in V for s in S), name="c14")
m.addConstrs((gp.quicksum(z[t, v, s] for v in V) == 1 for t in T for s in S), name="c15")

# (16)(17) 조건부 등식 big-M 선형화  ─ v1의 McCormick y (107,360행) 대체
m.addConstrs((g[t, s] - gp.quicksum(dist[v, w] * z[t, w, s + 1] for w in V) <= BIGM * (1 - z[t, v, s])
              for t in T for v in V for s in Smove), name="c16")
m.addConstrs((gp.quicksum(dist[v, w] * z[t, w, s + 1] for w in V) - g[t, s] <= BIGM * (1 - z[t, v, s])
              for t in T for v in V for s in Smove), name="c17")

# (18) 시즌 총 이동거리, (19)(20) 편차 한도
m.addConstrs((L[t] == gp.quicksum(g[t, s] for s in Smove) for t in T), name="c18")
m.addConstrs((L[t] - L[u] <= D_max for (t, u) in Tunorder), name="c19")
m.addConstrs((L[u] - L[t] <= D_max for (t, u) in Tunorder), name="c20")
print("⑥ 추가")

⑥ 추가


### 7) 변수 범위 — (22)~(26)

$$x_{ijs}\in\{0,1\}\qquad \forall (i,j)\in T^{order},\ s\in S \tag{22}$$

$$h_{ts}\in\{0,1\}\qquad \forall t\in T,\ s\in S \tag{23}$$

$$z_{tvs}\in\{0,1\}\qquad \forall t\in T,\ v\in V,\ s\in S \tag{24}$$

$$0\le g_{ts}\le M\qquad \forall t\in T,\ s\in S^{move} \tag{25}$$

$$L_t\ge 0\qquad \forall t\in T \tag{26}$$

In [12]:
m.update()
print("전체 모델 | 변수", m.NumVars, "| 제약", m.NumConstrs)
print("v1 대비  | 변수 44,200 -> %d (%.0f%%↓) | 제약 115,995 -> %d (%.0f%%↓)"
      % (m.NumVars, 100 * (1 - m.NumVars / 44200), m.NumConstrs, 100 * (1 - m.NumConstrs / 115995)))

전체 모델 | 변수 9000 | 제약 16555
v1 대비  | 변수 44,200 -> 9000 (80%↓) | 제약 115,995 -> 16555 (86%↓)


## (검증) 스케줄 추출 · 6조건 자동 검사
모델과 **독립적으로** 원자료(csv)만 보고 6개 조건을 재검사한다. 제출 직전 반드시 통과해야 한다.

In [13]:
def extract_schedule():
    rows = [{"series": int(s), "home_team_id": int(i), "away_team_id": int(j)}
            for (i, j) in Torder for s in S if x[i, j, s].X > 0.5]
    return (pd.DataFrame(rows).sort_values(["series", "home_team_id"])
            [["series", "home_team_id", "away_team_id"]].reset_index(drop=True))

def verify_schedule(df, verbose=True):
    mu = {s: [] for s in S}
    for r in df.itertuples(): mu[int(r.series)].append((int(r.home_team_id), int(r.away_team_id)))
    res, det = {}, {}
    res["① 편성기본"] = (len(df) == 225 and all(len(mu[s]) == 5 for s in S)
                     and all(sorted(t for (h_, a_) in mu[s] for t in (h_, a_)) == T for s in S))
    res["② 개막·올스타"] = (mu[1] and set(mu[1]) == set(mu[31]) and set(h_ for (h_, a_) in mu[1]) <= set(R))
    meet = Counter(frozenset((h_, a_)) for s in S for (h_, a_) in mu[s])
    c3 = all(meet[frozenset((i, j))] == n_meet for (i, j) in Tunorder)
    for (i, j) in Tunorder:
        for s in Scons:
            if sum(1 for sp in range(s, s + 4) for (h_, a_) in mu[sp] if {h_, a_} == {i, j}) > 1: c3 = False
    res["③ 대진 횟수·간격"] = c3
    def is_home(t, s): return any(h_ == t for (h_, a_) in mu[s])
    c4 = True
    for (i, j) in Torder:
        if sum(1 for s in B1 if (i, j) in mu[s]) != 1 or sum(1 for s in B2 if (i, j) in mu[s]) != 1: c4 = False
    for t in T:
        if not (H_lo <= sum(is_home(t, s) for s in S) <= H_hi): c4 = False
        if not (w_lo <= sum(is_home(t, s) for s in SW) <= w_hi): c4 = False
    res["④ 홈·원정 밸런스"] = c4
    c5 = True
    for t in T:
        for s in Scons:
            hc = sum(is_home(t, sp) for sp in range(s, s + 4))
            if hc > c_max or hc < 4 - c_max: c5 = False
    for s in S:
        if sum(is_home(t, s) for t in C) > 1: c5 = False
    res["⑤ 연속·구장중복"] = c5
    def vof(t, s):
        for (h_, a_) in mu[s]:
            if h_ == t: return ven[t]
            if a_ == t: return ven[h_]
    Ls = {t: sum(dist[vof(t, s), vof(t, s + 1)] for s in Smove) for t in T}
    dev = max(Ls.values()) - min(Ls.values())
    res["⑥ 이동거리 편차"] = dev <= D_max
    det["편차"] = round(dev, 1); det["팀별거리"] = {t: round(Ls[t]) for t in T}
    det["수익"] = sum(p[(int(r.home_team_id), int(r.away_team_id), int(r.series))] for r in df.itertuples())
    if verbose:
        for k, v in res.items(): print("  %-16s: %s" % (k, "O" if v else "X"))
        print("  수익:", det["수익"], "| 편차:", det["편차"], "km (한도", D_max, ") |",
              "통과" if all(res.values()) else "위반")
    return res, det

# 배선 확인: 예시 스케줄 = 75,798
ex = pd.read_csv(DATA_DIR / "phase2_example_schedule.csv", encoding="utf-8-sig")
print("[배선 확인] 예시 스케줄"); verify_schedule(ex)

[배선 확인] 예시 스케줄
  ① 편성기본          : O
  ② 개막·올스타        : O
  ③ 대진 횟수·간격      : O
  ④ 홈·원정 밸런스      : O
  ⑤ 연속·구장중복       : O
  ⑥ 이동거리 편차       : O
  수익: 75798.0 | 편차: 3970.0 km (한도 4500 ) | 통과


({'① 편성기본': True,
  '② 개막·올스타': True,
  '③ 대진 횟수·간격': True,
  '④ 홈·원정 밸런스': True,
  '⑤ 연속·구장중복': True,
  '⑥ 이동거리 편차': True},
 {'편차': 3970.0,
  '팀별거리': {1: 6495,
   2: 7470,
   3: 9455,
   4: 8740,
   5: 7235,
   6: 5665,
   7: 6585,
   8: 8645,
   9: 6905,
   10: 9635},
  '수익': 75798.0})

## 6. 모델 실행

**실행 규정 (컴페티션부 확인).**

- 실행 시간 제한 **8시간** → `TimeLimit = 28800`
- **이전 실행의 해를 저장해 다음 실행의 초기해로 재사용하는 것은 불가** → 외부 warm start 파일을 사용하지 않고,
  대회가 제공한 `phase2_example_schedule.csv`(75,798)만 초기해로 넣어 시작 즉시 실행가능해를 확보한다.

**파라미터 근거.** 정량평가는 **incumbent 목적값**만 보므로 bound 증명에 시간을 쓰면 손해다.
v1(McCormick) 실행은 마지막 incumbent 개선이 16,376초, 종료가 29,036초로 43%가 증명에 소모되었다.
반면 v2에서는 `NoRelHeurTime` 구간이 **6초 만에 v1의 8시간 결과를 추월**했고 개선의 대부분을 만들었다.
따라서 heuristic 예산을 1시간으로 늘리고 `MIPFocus=1`로 탐색을 해 발견 쪽에 집중시킨다.

| 파라미터 | 값 | 이유 |
|:--|:--|:--|
| `LogFile` | `승호_phase2_log.txt` | 정오표 1 (제출물) |
| `TimeLimit` | 28800 | 규정 8시간 |
| `MIPFocus` | 1 | 증명보다 좋은 해 탐색 |
| `NoRelHeurTime` | 3600 | B&B 전 heuristic — 이 모델에서 가장 생산적인 단계 |
| `ImproveStartGap` | 0.05 | gap 5% 이하부터 feasibility 모드 전환 |
| `Heuristics` / `RINS` | 0.35 / 25 | 이웃 탐색 강화 |
| `Threads` | 0 | 자동 (하드코딩 시 과다구독 위험) |

In [14]:
TEAM_NAME = "2A"
OUT_PATH  = f"{TEAM_NAME}_phase2_schedule.csv"
LOG_PATH  = f"{TEAM_NAME}_phase2_log.txt"   

# 스태프 확인사항: 이전 실행의 해를 저장해 초기해로 재사용하는 것은 불가.
# 따라서 외부 warm start 파일을 쓰지 않고, 대회가 제공한 예시 스케줄만 초기해로 사용한다.
warmset = {(int(r.home_team_id), int(r.away_team_id), int(r.series)) for r in ex.itertuples()}
print("초기해: 제공 예시 스케줄 | 수익", sum(p[k] for k in warmset))
for (i, j) in Torder:
    for s in S:
        x[i, j, s].Start = 1 if (i, j, s) in warmset else 0

# 개선될 때마다 CSV 즉시 저장 (중간 중단 대비)
_keys  = [(i, j, s) for (i, j) in Torder for s in S]
_xvars = [x[i, j, s] for (i, j, s) in _keys]
_best  = {"obj": float("-inf")}
def save_incumbent(model, where):
    if where == GRB.Callback.MIPSOL:
        obj = model.cbGet(GRB.Callback.MIPSOL_OBJ)
        if obj > _best["obj"] + 1e-6:
            _best["obj"] = obj
            xv = model.cbGetSolution(_xvars)
            rows = [(int(s), int(i), int(j)) for ((i, j, s), val) in zip(_keys, xv) if val > 0.5]
            (pd.DataFrame(rows, columns=["series", "home_team_id", "away_team_id"])
               .sort_values(["series", "home_team_id"])
               .to_csv(OUT_PATH, index=False, encoding="utf-8-sig"))
            print(f"   [자동저장] obj={obj:.1f} -> {OUT_PATH}")

m.setParam("LogFile", LOG_PATH)      
m.setParam("Seed", 0)
m.setParam("Threads", 0)             # 0 = 자동 (물리 코어 수)
m.setParam("MIPFocus", 1)            # 정량평가는 incumbent만 보므로 bound 증명보다 해 탐색 우선
m.setParam("NoRelHeurTime", 3600)    # B&B 전 heuristic 1시간 — 이 모델에서 개선의 대부분을 만든 단계
m.setParam("ImproveStartGap", 0.05)
m.setParam("Heuristics", 0.35)
m.setParam("RINS", 25)
m.setParam("TimeLimit", 28800)       # 스태프 확인: 실행 제한 8시간

print(">> 최적화 시작 (제한 8시간)")
m.optimize(save_incumbent)

sched = extract_schedule(); FINAL_OBJ = m.ObjVal
print(f">> 완료 | 목적값 {FINAL_OBJ:.1f} | MIPGap {m.MIPGap:.4f} | 소요 {m.Runtime:.0f}s")

초기해: 제공 예시 스케줄 | 수익 75798.0
Set parameter LogFile to value "2A_phase2_log.txt"
Set parameter Seed to value 0
Set parameter Threads to value 0
Set parameter MIPFocus to value 1
Set parameter NoRelHeurTime to value 3600
Set parameter ImproveStartGap to value 0.05
Set parameter Heuristics to value 0.35
Set parameter RINS to value 25
Set parameter TimeLimit to value 28800
>> 최적화 시작 (제한 8시간)
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 7 155H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 22 logical processors, using up to 22 threads

Non-default parameters:
TimeLimit  28800
Heuristics  0.35
MIPFocus  1
NoRelHeurTime  3600
RINS  25
ImproveStartGap  0.05

Academic license 2847604 - for non-commercial use only - registered to bi___@pusan.ac.kr
Optimize a model with 16555 rows, 9000 columns and 132435 nonzeros (Max)
Model fingerprint: 0x34bce829
Model has 4050 linear objective coefficients
Variable typ

## 7. 결과 저장 · 최종 검증

제출용 CSV를 저장하고, **모델과 독립적으로** 원자료만 보고 6개 조건을 재검사한다.

In [15]:
sched.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print("저장:", OUT_PATH, "| 행수", len(sched), "| 총수익", round(FINAL_OBJ, 1),
      "| 예시(75,798) 대비 +", round(FINAL_OBJ - 75798, 1))

print("\n[최종 스케줄 6조건 검증]")
fres, fdet = verify_schedule(sched)
assert all(fres.values()), "일부 조건 위반 — 확인 필요"
print("\n팀별 총 이동거리:", fdet["팀별거리"])
display(sched.head(10))

저장: 2A_phase2_schedule.csv | 행수 225 | 총수익 79023.0 | 예시(75,798) 대비 + 3225.0

[최종 스케줄 6조건 검증]
  ① 편성기본          : O
  ② 개막·올스타        : O
  ③ 대진 횟수·간격      : O
  ④ 홈·원정 밸런스      : O
  ⑤ 연속·구장중복       : O
  ⑥ 이동거리 편차       : O
  수익: 79023.0 | 편차: 4025.0 km (한도 4500 ) | 통과

팀별 총 이동거리: {1: 6610, 2: 5775, 3: 8905, 4: 9800, 5: 6525, 6: 6875, 7: 5975, 8: 8030, 9: 7745, 10: 8390}


,series,home_team_id,away_team_id
0,1,1,7
1,1,3,8
2,1,4,10
3,1,5,2
4,1,6,9
5,2,1,3
6,2,4,6
7,2,5,8
8,2,7,2
9,2,9,10


### 조건–제약식 매핑표 *(docx와 동일)*

| 조건 | 규칙 요약 | 제약식 |
|:--|:--|:--|
| ① 시리즈 편성 기본 | 매 시리즈 10팀이 5개 매치업으로 빠짐없이 편성 | (1), (2) |
| ② 개막 · 올스타 | 개막 홈 개최권 상위 5팀, 31번 = 1번 재현 | (3), (4), (21) |
| ③ 매치업 횟수 · 간격 | 5회 대결, 연속 4시리즈 창에 최대 1회 | (5), (6) |
| ④ 홈 · 원정 배분과 총량 | 1–18 / 19–36 각 1회씩, 시즌 홈 22~23 (주말 11~12) | (7), (8), (9), (10) |
| ⑤ 연속 제한 · 구장 사용 | 홈/원정 4연속 불가, Metro_Field 동시 홈 불가 | (11), (12), (13) |
| ⑥ 이동거리 편차 | 시리즈 간 이동만 계산, 구단 간 총거리 차 ≤ 4500 | (14)~(20) |

*(22)~(26)은 변수 범위 선언.*

---
### 제출 체크리스트
- [ ] `비원_Phase2_schedule.csv` — 225행, 6조건 전부 O
- [ ] `비원_phase2_log.txt` — 로그 내부 파일명이 제출 파일명과 일치
- [ ] `.docx` 제약식 번호가 (1)~(26)로 본 노트북과 1:1 일치
- [ ] Restart & Run All 후 CSV · 목적값 · log가 **같은 한 번의 실행 결과**